# 03 — Volatility Estimator Comparison

> ¿La elección del estimador de volatilidad cambia materialmente el sizing, el control de riesgo y el perfil retorno-riesgo de una misma señal TSMOM?

La señal permanece fija. El caso principal es SPY con TSMOM 21d long/cash; la comparación se repite de forma compacta para SPY 252d, QQQ 21d, XLE 21d y GLD 21d.


## <b>Diseño causal</b>

Los retornos hasta `t` producen el estimador de volatilidad en `t`; junto con la señal TSMOM en `t`, fijan el peso $w_t$. Ese peso sólo se aplica al retorno de `t+1`: $R_{t+1}^{strategy} = w_t r_{t+1}$.

EWMA usa $λ = 0.94$ y rolling volatility usa 21 sesiones. Ambos comparten $σ^* = 12%$ anual y $L_{max} = 1.0$. En long/cash, una señal positiva recibe exposición escalada y cualquier otra recibe peso cero.


In [1]:
from datetime import date
from pathlib import Path
import math

import plotly.graph_objects as go
import polars as pl

SYMBOLS = ("SPY", "QQQ", "XLE", "GLD")
RESEARCH_CASES = (("SPY", 21), ("SPY", 252), ("QQQ", 21), ("XLE", 21), ("GLD", 21))
WINDOW_START = date(2018, 1, 2)
WINDOW_END = date(2026, 11, 17)
LAMBDA = 0.94
ROLLING_WINDOW = 21
WARMUP_RETURNS = 20
TARGET_VOL = 0.12
MAX_LEVERAGE = 1.0
TRADING_DAYS = 252

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        for project_root in (candidate, candidate / "Momentum_systematic_strategies"):
            if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
                return project_root
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf desde el directorio actual.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"


## <b>Gold daily data and fixed signals</b>

> Esta notebook lee exclusivamente `data/gold/daily_series/etf/<SYMBOL>.parquet`. Conserva la señal clásica por signo del momentum return y no modifica las notebooks anteriores.


In [2]:
frames = {}
for symbol in SYMBOLS:
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")
    frames[symbol] = (
        pl.read_parquet(path)
        .with_columns(pl.col("session_date").cast(pl.Date))
        .filter(pl.col("session_date").is_between(WINDOW_START, WINDOW_END, closed="both"))
        .select("session_date", "adjusted_close")
        .sort("session_date")
    )

prices = frames["SPY"].rename({"adjusted_close": "SPY"})
for symbol in SYMBOLS[1:]:
    prices = prices.join(frames[symbol].rename({"adjusted_close": symbol}), on="session_date", how="inner")

prices = prices.sort("session_date")
{
    "rows": prices.height,
    "start": prices.get_column("session_date").min(),
    "end": prices.get_column("session_date").max(),
    "nulls": int(prices.null_count().sum_horizontal()[0]),
}


{'rows': 2232,
 'start': datetime.date(2018, 1, 2),
 'end': datetime.date(2026, 11, 17),
 'nulls': 0}

In [3]:
returns_panel = prices.with_columns([
    pl.col(symbol).log().diff().alias(f"{symbol}_ret") for symbol in SYMBOLS
])

feature_panel = returns_panel.with_columns([
    (pl.col(symbol).log() - pl.col(symbol).shift(horizon).log()).alias(f"{symbol}_mom_{horizon}")
    for symbol, horizon in RESEARCH_CASES
])

feature_panel = feature_panel.with_columns([
    pl.when(pl.col(f"{symbol}_mom_{horizon}").is_null()).then(None)
    .when(pl.col(f"{symbol}_mom_{horizon}") > 0).then(1)
    .when(pl.col(f"{symbol}_mom_{horizon}") < 0).then(-1)
    .otherwise(0).cast(pl.Int8).alias(f"{symbol}_sig_{horizon}")
    for symbol, horizon in RESEARCH_CASES
])

feature_panel.select("session_date", "SPY_ret", "SPY_mom_21", "SPY_sig_21").tail()


session_date,SPY_ret,SPY_mom_21,SPY_sig_21
date,f64,f64,i8
2026-11-11,0.013102,-0.023671,-1
2026-11-12,-0.003547,-0.016349,-1
2026-11-13,-0.003639,-0.02283,-1
2026-11-16,-0.0193,-0.031003,-1
2026-11-17,-0.010201,-0.045427,-1


## <b>Causal volatility estimates</b>

EWMA y la desviación estándar rolling incluyen únicamente retornos observados hasta la fecha de la fila. Ninguno usa estadísticos de muestra completa ni retornos futuros.


In [4]:
estimator_columns = []
for symbol in SYMBOLS:
    returns = feature_panel.get_column(f"{symbol}_ret").to_list()
    variance = [None] * len(returns)
    first_valid = next(index for index, value in enumerate(returns) if value is not None)
    initial = first_valid + WARMUP_RETURNS - 1
    variance[initial] = sum(value ** 2 for value in returns[first_valid:initial + 1]) / WARMUP_RETURNS
    for index in range(initial + 1, len(returns)):
        variance[index] = LAMBDA * variance[index - 1] + (1 - LAMBDA) * returns[index] ** 2
    estimator_columns.append(pl.Series(
        f"{symbol}_ewma_vol_ann",
        [None if value is None else math.sqrt(value * TRADING_DAYS) for value in variance],
    ))

feature_panel = feature_panel.with_columns(estimator_columns).with_columns([
    (pl.col(f"{symbol}_ret").rolling_std(ROLLING_WINDOW) * math.sqrt(TRADING_DAYS)).alias(
        f"{symbol}_rolling_vol_ann"
    )
    for symbol in SYMBOLS
])
feature_panel.select("session_date", "SPY_ewma_vol_ann", "SPY_rolling_vol_ann").tail()


session_date,SPY_ewma_vol_ann,SPY_rolling_vol_ann
date,f64,f64
2026-11-11,0.214733,0.223148
2026-11-12,0.208647,0.220547
2026-11-13,0.202785,0.220349
2026-11-16,0.210444,0.226767
2026-11-17,0.207854,0.227702


### <b>SPY forecast distribution</b>

> Estas estadísticas describen el comportamiento de los estimadores, no la calidad de una estrategia.


In [5]:
def lag1_correlation(values):
    return pl.DataFrame({"current": values[1:], "lag": values[:-1]}).select(
        pl.corr("current", "lag")
    ).item()

forecast_rows = []
for label, column in [("EWMA 0.94", "SPY_ewma_vol_ann"), ("Rolling 21d", "SPY_rolling_vol_ann")]:
    values = feature_panel.get_column(column).drop_nulls().to_list()
    series = pl.Series(values)
    forecast_rows.append({
        "estimator": label,
        "mean (%)": series.mean() * 100,
        "median (%)": series.median() * 100,
        "p10 (%)": series.quantile(0.10) * 100,
        "p90 (%)": series.quantile(0.90) * 100,
        "max (%)": series.max() * 100,
        "lag 1 autocorrelation": lag1_correlation(values),
    })

forecast_summary = pl.DataFrame(forecast_rows).with_columns(pl.all().exclude("estimator").round(3))
forecast_summary


estimator,mean (%),median (%),p10 (%),p90 (%),max (%),lag 1 autocorrelation
str,f64,f64,f64,f64,f64,f64
"""EWMA 0.94""",16.517,13.664,9.46,25.489,81.371,0.994
"""Rolling 21d""",16.079,13.419,8.181,25.881,93.655,0.992


In [6]:
figure = go.Figure()
for column, label in [("SPY_ewma_vol_ann", "EWMA 0.94"), ("SPY_rolling_vol_ann", "Rolling 21d")]:
    figure.add_scatter(x=feature_panel["session_date"], y=feature_panel[column] * 100, mode="lines", name=label)

figure.update_layout(
    title="SPY — causal annualized volatility estimates",
    yaxis_title="annualized volatility (%)",
    legend_title_text="",
)
figure.show()


## <b>Fixed-signal volatility scaling</b>

La regla es idéntica para ambos estimadores: $w_t = s_t min(σ^*/σ̂_t, L_{max})$. Para long/cash se sustituye cualquier señal no positiva por peso cero; sólo cambia la estimación de $σ̂_t$.


In [7]:
strategy_panel = feature_panel.with_columns([
    pl.col(f"{symbol}_ret").shift(-1).alias(f"{symbol}_next_ret") for symbol in SYMBOLS
])

weight_expressions = []
for symbol, horizon in RESEARCH_CASES:
    signal = pl.col(f"{symbol}_sig_{horizon}")
    raw_weight = pl.when(signal.is_null()).then(None).when(signal == 1).then(1.0).otherwise(0.0)
    for estimator, column in [("ewma", f"{symbol}_ewma_vol_ann"), ("rolling", f"{symbol}_rolling_vol_ann")]:
        scale = pl.min_horizontal(pl.lit(TARGET_VOL) / pl.col(column), pl.lit(MAX_LEVERAGE))
        weight_expressions.append(
            pl.when(signal.is_null()).then(None).when(signal == 1).then(scale).otherwise(0.0)
            .alias(f"{symbol}_w_{estimator}_{horizon}")
        )
    weight_expressions.append(raw_weight.alias(f"{symbol}_w_raw_{horizon}"))

strategy_panel = strategy_panel.with_columns(weight_expressions).with_columns([
    (pl.col(f"{symbol}_w_{variant}_{horizon}") * pl.col(f"{symbol}_next_ret")).alias(
        f"{symbol}_strat_{variant}_{horizon}"
    )
    for symbol, horizon in RESEARCH_CASES
    for variant in ("raw", "ewma", "rolling")
])

required = [f"{symbol}_next_ret" for symbol in SYMBOLS] + [
    f"{symbol}_strat_{variant}_{horizon}"
    for symbol, horizon in RESEARCH_CASES
    for variant in ("raw", "ewma", "rolling")
]
comparison_panel = strategy_panel.drop_nulls(required)
{
    "start": comparison_panel.get_column("session_date").min(),
    "end": comparison_panel.get_column("session_date").max(),
    "observations": comparison_panel.height,
}


{'start': datetime.date(2019, 1, 3),
 'end': datetime.date(2026, 11, 16),
 'observations': 1979}

In [8]:
comparison_panel.select(
    "session_date", "SPY_sig_21", "SPY_ewma_vol_ann", "SPY_rolling_vol_ann",
    "SPY_w_ewma_21", "SPY_w_rolling_21", "SPY_next_ret",
    "SPY_strat_ewma_21", "SPY_strat_rolling_21",
).head(5)


session_date,SPY_sig_21,SPY_ewma_vol_ann,SPY_rolling_vol_ann,SPY_w_ewma_21,SPY_w_rolling_21,SPY_next_ret,SPY_strat_ewma_21,SPY_strat_rolling_21
date,i8,f64,f64,f64,f64,f64,f64,f64
2019-01-03,-1,0.27939,0.291404,0.0,0.0,0.032947,0.0,0.0
2019-01-04,-1,0.299646,0.314505,0.0,0.0,0.007854,0.0,0.0
2019-01-07,-1,0.292118,0.299482,0.0,0.0,0.009352,0.0,0.0
2019-01-08,-1,0.285544,0.30233,0.0,0.0,0.004663,0.0,0.0
2019-01-09,-1,0.277438,0.29277,0.0,0.0,0.003521,0.0,0.0


## <b>Economic comparison</b>

> Raw long/cash, EWMA target and rolling target usan la misma ventana efectiva dentro de cada caso. La exposición es el valor absoluto del peso diario.


In [9]:
def realized_volatility_metrics(log_returns):
    realized = pl.Series(log_returns).rolling_std(ROLLING_WINDOW).drop_nulls() * math.sqrt(TRADING_DAYS)
    return {
        "realized vol mean (%)": realized.mean() * 100,
        "realized vol median (%)": realized.median() * 100,
        "realized vol MAE vs 12%": (realized - TARGET_VOL).abs().mean() * 100,
        "realized vol within 10–14% (%)": realized.is_between(0.10, 0.14, closed="both").mean() * 100,
    }

def performance_metrics(log_returns, exposures):
    returns = pl.Series(log_returns)
    weights = pl.Series([abs(value) for value in exposures])
    wealth = returns.cum_sum().exp()
    drawdown = wealth / wealth.cum_max() - 1
    daily_volatility = returns.std()
    return {
        "annualized return (%)": (wealth[-1] ** (TRADING_DAYS / len(returns)) - 1) * 100,
        "annualized vol (%)": daily_volatility * math.sqrt(TRADING_DAYS) * 100,
        "Sharpe (rf=0)": returns.mean() / daily_volatility * math.sqrt(TRADING_DAYS),
        "max drawdown (%)": drawdown.min() * 100,
        "average exposure (%)": weights.mean() * 100,
        "median exposure (%)": weights.median() * 100,
    }

metric_rows = []
for symbol, horizon in RESEARCH_CASES:
    for variant, label in [("raw", "raw long/cash"), ("ewma", "EWMA vol target"), ("rolling", "rolling 21d vol target")]:
        returns = comparison_panel.get_column(f"{symbol}_strat_{variant}_{horizon}").to_list()
        weights = comparison_panel.get_column(f"{symbol}_w_{variant}_{horizon}").to_list()
        metric_rows.append({
            "asset": symbol, "horizon": f"{horizon}d", "variant": label,
            **performance_metrics(returns, weights),
            **realized_volatility_metrics(returns),
        })

metrics_summary = pl.DataFrame(metric_rows).with_columns(
    pl.all().exclude(["asset", "horizon", "variant"]).round(3)
)
metrics_summary.filter((pl.col("asset") == "SPY") & pl.col("horizon").is_in(["21d", "252d"]))


asset,horizon,variant,annualized return (%),annualized vol (%),Sharpe (rf=0),max drawdown (%),average exposure (%),median exposure (%),realized vol mean (%),realized vol median (%),realized vol MAE vs 12%,realized vol within 10–14% (%)
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""SPY""","""21d""","""raw long/cash""",10.62,11.341,0.89,-18.499,69.277,100.0,10.169,9.853,4.144,33.18
"""SPY""","""21d""","""EWMA vol target""",7.568,8.641,0.844,-12.222,58.613,75.389,8.009,8.08,4.231,26.901
"""SPY""","""21d""","""rolling 21d vol target""",8.36,8.855,0.907,-12.275,59.726,77.853,8.191,8.394,4.131,27.565
"""SPY""","""252d""","""raw long/cash""",9.776,15.069,0.619,-31.168,84.639,100.0,12.55,11.587,5.464,32.261
"""SPY""","""252d""","""EWMA vol target""",8.935,10.887,0.786,-14.879,72.071,88.116,9.746,10.327,3.951,36.702
"""SPY""","""252d""","""rolling 21d vol target""",9.702,10.977,0.844,-14.516,72.663,88.954,9.875,10.219,3.888,36.039


## <b>SPY 21d exposure</b>

> Esta figura deja visible cómo una misma señal long/cash recibe pesos distintos cuando cambia sólo el estimador de volatilidad.


In [10]:
figure = go.Figure()
for column, label in [("SPY_w_ewma_21", "EWMA 0.94"), ("SPY_w_rolling_21", "Rolling 21d")]:
    figure.add_scatter(x=comparison_panel["session_date"], y=comparison_panel[column], mode="lines", name=label)

figure.update_layout(
    title="SPY 21d long/cash — volatility-targeted exposure",
    yaxis_title="weight",
    legend_title_text="",
)
figure.show()


### <b>SPY target accuracy</b>

La volatilidad realizada es rolling de 21 sesiones y anualizada. El MAE mide distancia absoluta frente a 12%; el intervalo 10–14% es una banda diagnóstica, no una regla de optimización.


In [11]:
target_accuracy = metrics_summary.filter(
    (pl.col("asset") == "SPY") &
    (pl.col("horizon") == "21d") &
    pl.col("variant").is_in(["EWMA vol target", "rolling 21d vol target"])
).select(
    "variant", "realized vol mean (%)", "realized vol median (%)",
    "realized vol MAE vs 12%", "realized vol within 10–14% (%)",
)
target_accuracy


variant,realized vol mean (%),realized vol median (%),realized vol MAE vs 12%,realized vol within 10–14% (%)
str,f64,f64,f64,f64
"""EWMA vol target""",8.009,8.08,4.231,26.901
"""rolling 21d vol target""",8.191,8.394,4.131,27.565


## <b>Selected high-volatility episodes</b>

Se usan dos periodos fijos ya presentes en la muestra, sin buscar fechas que favorezcan a un estimador. La tabla resume el pico estimado y la menor exposición resultante; las sesiones de recuperación cuentan desde el pico hasta que el factor de escala vuelve a 90% o más.


In [12]:
shock_periods = {
    "2020-02 to 2020-04": (date(2020, 2, 20), date(2020, 4, 15)),
    "2022-09 to 2022-10": (date(2022, 9, 1), date(2022, 10, 31)),
}

shock_rows = []
for episode, (start, end) in shock_periods.items():
    window = comparison_panel.filter(pl.col("session_date").is_between(start, end, closed="both"))
    for estimator in ("ewma", "rolling"):
        vol_column = f"SPY_{estimator}_vol_ann"
        weight_column = f"SPY_w_{estimator}_21"
        peak_index = window.get_column(vol_column).arg_max()
        peak_date = window.get_column("session_date")[peak_index]
        future = comparison_panel.filter(pl.col("session_date") >= peak_date).get_column(weight_column).to_list()
        recovery = next((index for index, value in enumerate(future) if value >= 0.90), None)
        shock_rows.append({
            "episode": episode,
            "estimator": estimator,
            "peak estimated vol (%)": window.get_column(vol_column).max() * 100,
            "minimum exposure (%)": window.get_column(weight_column).min() * 100,
            "sessions to 90% exposure": recovery,
        })

shock_summary = pl.DataFrame(shock_rows).with_columns(
    pl.col(["peak estimated vol (%)", "minimum exposure (%)"]).round(3)
)
shock_summary


episode,estimator,peak estimated vol (%),minimum exposure (%),sessions to 90% exposure
str,str,f64,f64,i64
"""2020-02 to 2020-04""","""ewma""",81.371,0.0,107
"""2020-02 to 2020-04""","""rolling""",93.655,0.0,80
"""2022-09 to 2022-10""","""ewma""",28.237,0.0,126
"""2022-09 to 2022-10""","""rolling""",29.23,0.0,120


## <b>Cross-asset summary</b>

La tabla conserva sólo EWMA y rolling para los cinco casos definidos. Todas las columnas se calculan sobre la misma ventana efectiva.


In [13]:
cross_asset_summary = metrics_summary.filter(
    pl.col("variant").is_in(["EWMA vol target", "rolling 21d vol target"])
).select(
    "asset", "horizon", "variant", "annualized return (%)", "annualized vol (%)",
    "Sharpe (rf=0)", "max drawdown (%)", "average exposure (%)",
    "realized vol MAE vs 12%",
)
cross_asset_summary


asset,horizon,variant,annualized return (%),annualized vol (%),Sharpe (rf=0),max drawdown (%),average exposure (%),realized vol MAE vs 12%
str,str,str,f64,f64,f64,f64,f64,f64
"""SPY""","""21d""","""EWMA vol target""",7.568,8.641,0.844,-12.222,58.613,4.231
"""SPY""","""21d""","""rolling 21d vol target""",8.36,8.855,0.907,-12.275,59.726,4.131
"""SPY""","""252d""","""EWMA vol target""",8.935,10.887,0.786,-14.879,72.071,3.951
"""SPY""","""252d""","""rolling 21d vol target""",9.702,10.977,0.844,-14.516,72.663,3.888
"""QQQ""","""21d""","""EWMA vol target""",8.222,9.305,0.849,-17.698,44.995,4.202
"""QQQ""","""21d""","""rolling 21d vol target""",9.86,9.828,0.957,-17.351,47.624,4.097
"""XLE""","""21d""","""EWMA vol target""",3.096,9.263,0.329,-23.434,30.635,4.092
"""XLE""","""21d""","""rolling 21d vol target""",2.739,9.752,0.277,-25.109,32.059,3.938
"""GLD""","""21d""","""EWMA vol target""",7.722,9.964,0.747,-15.71,46.36,4.622


## <b>Interpretación</b>

- En SPY, ambos forecasts son muy persistentes (autocorrelación lag 1 de 0.994 para EWMA y 0.992 para rolling); rolling alcanza picos algo mayores alrededor de los shocks.
- Los pesos medios de SPY 21d son cercanos (58.6% EWMA y 59.7% rolling), aunque rolling recupera 90% de exposición antes en los dos episodios seleccionados.
- Rolling tiene un MAE ligeramente menor frente al target en SPY 21d (4.13 pp frente a 4.23 pp), pero ambas estrategias realizan cerca de 8% porque long/cash pasa parte del tiempo en efectivo.
- Rolling mejora modestamente el Sharpe en SPY y QQQ, mientras EWMA lo conserva mejor en XLE y GLD; ningún resultado aislado selecciona un estimador.


## <b>Conclusión</b>

- EWMA y rolling 21d producen forecasts muy persistentes y exposiciones medias cercanas para la señal fija.
- En SPY 21d, rolling reduce levemente el MAE frente al target, pero la volatilidad realizada de la estrategia queda cerca de 8% por los días en efectivo.
- Ambos targets reducen drawdown frente a long/cash raw; las diferencias de Sharpe son pequeñas en SPY y cambian de signo entre los demás activos.
- El análisis no incorpora costes, ejecución, combinaciones de activos ni nuevos modelos de señal.
- No aparece una dominancia robusta de un estimador entre activos y horizontes.

La evidencia descriptiva no es suficiente para priorizar GARCH todavía; antes conviene entender el desvío del target inducido por los periodos long/cash.
